## Triển khai trình tạo văn bản sử dụng mạng nơ-ron hồi quy (RNN)

Trong phần này, chúng ta sẽ tạo một trình tạo văn bản dựa trên ký tự sử dụng Mạng thần kinh hồi quy (RNN) trong TensorFlow và Keras. Chúng ta sẽ triển khai một RNN học các mẫu từ một chuỗi văn bản để tạo ra văn bản mới từng ký tự một.

In [1]:
#1. Nhập các thư viện cần thiết
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense


### Chi tiết về các thư viện sử dụng:
* **`numpy`**: Thư viện dùng để xử lý mảng và ma trận dữ liệu hiệu quả.
* **`tensorflow`**: Thư viện học sâu chính hỗ trợ việc xây dựng và huấn luyện mô hình.
* **`SimpleRNN`**: Lớp mạng nơ-ron hồi quy đơn giản trong Keras dùng để xử lý dữ liệu dạng chuỗi.
* **`Dense`**: Lớp mạng kết nối đầy đủ (fully connected) dùng làm lớp đầu ra để dự đoán ký tự tiếp theo.


## 2. Xác định văn bản đầu vào và chuẩn bị bộ ký tự

Xác định văn bản đầu vào và nhận diện các ký tự duy nhất trong văn bản đó, sau đó sẽ mã hóa chúng cho mô hình của mình.

In [3]:
# 2.Xác định văn bản đầu vào và chuẩn bị bộ ký tự
text = "This is GeeksforGeeks a software training institute"
chars = sorted(list(set(text)))
char_to_index = {char: i for i, char in enumerate(chars)}
index_to_char = {i: char for i, char in enumerate(chars)}


## Giải thích bước chuẩn bị dữ liệu:
* Chúng ta định nghĩa một câu văn bản mẫu `text` làm dữ liệu huấn luyện.
* `chars` lưu trữ danh sách tất cả các ký tự độc nhất xuất hiện trong văn bản, sắp xếp theo thứ tự.
* Hai từ điển `char_to_index` và `index_to_char` được tạo ra để ánh xạ qua lại giữa ký tự và chỉ số số nguyên (index) tương ứng của nó.


## 3. Tạo chuỗi và nhãn

Để huấn luyện RNN, chúng ta cần các chuỗi có độ dài cố định ( seq_length) và ký tự theo sau mỗi chuỗi làm nhãn.

In [4]:
seq_length = 3
sequences = []
labels = []

for i in range(len(text) - seq_length):
    seq = text[i:i + seq_length]
    label = text[i + seq_length]
    sequences.append([char_to_index[char] for char in seq])
    labels.append(char_to_index[label])

X = np.array(sequences)
y = np.array(labels)


### Giải thích bước tạo chuỗi và nhãn:
* `seq_length = 3` nghĩa là mô hình sẽ nhìn vào 3 ký tự liên tiếp để dự đoán ký tự thứ 4 tiếp theo.
* Đoạn mã trên sẽ cắt văn bản thành các đoạn nhỏ có độ dài bằng 3 lưu vào `sequences` (được số hóa qua chỉ số), và ký tự kế tiếp sẽ được lưu làm nhãn trong label.

## 4. Chuyển đổi chuỗi và nhãn sang mã hóa One-Hot
Để huấn luyện, chúng ta chuyển đổi Xvà ythành các tensor được mã hóa one-hot.

In [5]:
X_one_hot = tf.one_hot(X, len(chars))
y_one_hot = tf.one_hot(y, len(chars))


### Giải thích One-Hot Encoding:
* Do mô hình phân loại đa lớp hoạt động tốt nhất trên xác suất, chúng ta biến đổi đầu vào và nhãn thành dạng vector nhị phân (One-Hot). Kích thước của mỗi vector bằng số lượng ký tự duy nhất (`len(chars)`).


##5. Xây dựng mô hình RNN
Chúng tôi tạo một mô hình RNN đơn giản với lớp ẩn gồm 50 đơn vị và lớp đầu ra Dense với hàm kích hoạt softmax .

In [6]:
model = Sequential()
model.add(SimpleRNN(50, input_shape=(seq_length, len(chars)), activation='relu'))
model.add(Dense(len(chars), activation='softmax'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



### Cấu trúc mô hình:
* **`SimpleRNN(50, ...)`**: Lớp RNN đơn giản có 50 đơn vị ẩn, sử dụng hàm kích hoạt `relu`.
* **`Dense(len(chars), activation='softmax')`**: Lớp đầu ra có số lượng nơ-ron bằng kích thước bộ ký tự, hàm kích hoạt `softmax` giúp chuyển đổi kết quả đầu ra thành phân phối xác suất của các ký tự.


## 6. Biên soạn và huấn luyện mô hình
Biên dịch mô hình bằng cách sử dụng hàm categorical_crossentropymất mát và huấn luyện nó trong 100 epoch.

In [7]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_one_hot, y_one_hot, epochs=100)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.0417 - loss: 2.8381
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.0625 - loss: 2.8110
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0833 - loss: 2.7875
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.1042 - loss: 2.7652
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.1042 - loss: 2.7440
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1042 - loss: 2.7227
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.1458 - loss: 2.7021
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.1875 - loss: 2.6828
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.1875 - loss: 2.6625
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1875 - loss: 2.6420
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2292 - loss: 2.6219
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2500 - lo


### Biên dịch và huấn luyện:
* Mô hình sử dụng thuật toán tối ưu `adam` và hàm mất mát `categorical_crossentropy` (thường dùng cho bài toán phân loại đa lớp với nhãn dạng One-Hot).
* Tiến hành huấn luyện mô hình qua 100 chu kỳ (`epochs=100`) trên dữ liệu đã được chuẩn bị.


## 7. Tạo văn bản mới bằng cách sử dụng mô hình đã được huấn luyện
Sau quá trình huấn luyện, chúng tôi sử dụng một chuỗi khởi đầu để tạo ra văn bản mới từng ký tự một.

In [8]:
start_seq = "This is G"
generated_text = start_seq

for i in range(50):
    x = np.array([[char_to_index[char] for char in generated_text[-seq_length:]]])
    x_one_hot = tf.one_hot(x, len(chars))
    prediction = model.predict(x_one_hot)
    next_index = np.argmax(prediction)
    next_char = index_to_char[next_index]
    generated_text += next_char

print("Generated Text:")
print(generated_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━


### Quy trình tạo văn bản tự động:
1. Bắt đầu với một chuỗi khởi đầu: `This is G`.
2. Lấy 3 ký tự cuối cùng của chuỗi hiện tại để làm đầu vào dự đoán.
3. Mã hóa One-hot đầu vào này rồi đưa qua mô hình huấn luyện để dự đoán xác suất cho ký tự kế tiếp.
4. Tìm ký tự có xác suất cao nhất bằng hàm `np.argmax()` và ghép nó vào chuỗi kết quả.
5. Lặp lại quy trình trên 50 lần để tạo ra văn bản hoàn chỉnh.
